# Lab 3-02: Build a Production RAG Chatbot on NVIDIA Endpoints

**Chapter 3: Software Development for Generative AI and LLMs**  
**Lab Type:** Hosted-API Lab | **Estimated Time:** 60 minutes  
**Exam Domains Covered:** Software Development (24%), Experimentation (22%), Trustworthy AI (10%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Explain **Retrieval-Augmented Generation (RAG)** and when to choose RAG over **Fine-Tuning (PEFT/LoRA)** or **Prompt Engineering alone**.
2. Implement the complete two-stage NVIDIA RAG architecture from **Chapter 3**:
   - **Document Ingestion & Chunking** (`RecursiveCharacterTextSplitter` with `chunk_size` and `chunk_overlap`)
   - **Dense Embedding & Indexing** (`NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5")` + `FAISS`)
   - **Two-Stage Retrieval + Cross-Encoder Reranking** (`NVIDIARerank(model="nvidia/nv-rerankqa-mistral-4b-v3")`)
   - **Grounded Generation with Source Citations `[doc-id]`** (`ChatNVIDIA(model="meta/llama-3.1-8b-instruct")`)
3. Evaluate the RAG pipeline on both **in-scope** enterprise queries and **out-of-scope (unanswerable)** queries to verify hallucination prevention.

---
## 1. Conceptual Foundation: RAG vs. Fine-Tuning & Two-Stage Retrieval

In **Chapter 3 (*Retrieval-Augmented Generation (RAG) Architecture* & *RAG vs. Fine-Tuning*)**, the book provides a crucial decision matrix tested heavily on the NCA-GENL exam:

| Requirement / Scenario | Best Approach | Why? |
| :--- | :--- | :--- |
| **Frequently changing facts** (daily policy updates, live inventory, internal wiki) | **RAG** | Updating a vector index takes seconds and requires zero model retraining |
| **Strict source attribution / citations (`[doc-id]`) & auditability** | **RAG** | Retrieved chunks provide verifiable provenance for every claim |
| **Reducing hallucinations on proprietary domain facts** | **RAG** | Grounds generation in explicit retrieved evidence ("Answer ONLY from context") |
| **Teaching a new output style, tone, domain jargon, or compact task format** | **Fine-Tuning (LoRA)** | Modifies internal model weights to consistently follow a specialized behavior or schema |
| **Domain-specific jargon + live changing company documents** | **Hybrid (Fine-Tuning + RAG)** | Fine-tune embeddings/LLM on domain terminology, then use RAG for live facts |

### Why Add a Reranker (`NVIDIARerank`) After Vector Search?
1. **Stage 1 — Bi-Encoder Vector Retrieval (`NVIDIAEmbeddings` + `FAISS`):** Encodes queries and documents independently into vectors and retrieves top-$k$ candidates (e.g., $k=10$) in milliseconds via Approximate Nearest Neighbor (ANN) search. Fast, but compresses an entire chunk into a single vector.
2. **Stage 2 — Cross-Encoder Reranking (`NVIDIARerank`):** Jointly attends across the concatenated `(query, chunk)` pair using full Transformer self-attention to compute a fine-grained relevance score, selecting the top-$n$ (e.g., `top_n=3`) most relevant chunks for the LLM context window.

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import os
import re
import time
import math
import json
from dataclasses import dataclass
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd

API_KEY_AVAILABLE = bool(os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"))
print(f"Live NVIDIA NIM Endpoints Active: {API_KEY_AVAILABLE}")
if not API_KEY_AVAILABLE:
    print("-> Running with local dense/lexical embedding + cross-encoder reranker simulation so every RAG stage executes cleanly!")

Live NVIDIA NIM Endpoints Active: False
-> Running with local dense/lexical embedding + cross-encoder reranker simulation so every RAG stage executes cleanly!


---
## 2. Step 1: Enterprise Knowledge Base & Overlapping Document Chunking

Why do we set `chunk_overlap > 0` (typically 10%–20% of `chunk_size`) when splitting documents?
- If `chunk_overlap = 0`, a hard boundary can slice a critical sentence or table row in half—separating a subject in Chunk $i$ from its numerical specification in Chunk $i+1$.
- Overlapping chunks ensure that boundary-spanning facts remain intact inside at least one chunk.

Let's create a realistic multi-document NVIDIA Enterprise AI knowledge base and chunk it using `RecursiveCharacterTextSplitter` (with a built-in recursive splitter fallback if `langchain_text_splitters` is not installed).

In [2]:
@dataclass
class DocChunk:
    doc_id: str
    title: str
    chunk_id: str
    text: str
    char_len: int


KNOWLEDGE_BASE_DOCS = [
    {
        "doc_id": "NIM-OPS-101",
        "title": "NVIDIA NIM Deployment & Port Configuration Policy",
        "text": (
            "NVIDIA NIM containers expose an OpenAI-compatible HTTP REST server on default port 8000 "
            "and health check endpoints at /v1/health/ready and /v1/health/live. When launching a NIM "
            "container with Docker, engineers must pass -e NGC_API_KEY, --gpus all, and mount a local "
            "cache directory to /opt/nim/.cache so TensorRT-LLM engines are cached across container restarts. "
            "For Llama-3-8B-Instruct in FP16 precision, a minimum of 1x NVIDIA L4 (24GB) or A10G GPU is required."
        ),
    },
    {
        "doc_id": "RAG-ARCH-204",
        "title": "Enterprise RAG Retrieval & Reranking Standards",
        "text": (
            "All production RAG pipelines must use the two-stage retrieval pattern: first retrieve the top-10 "
            "candidate chunks from the FAISS or Milvus vector index using nvidia/nv-embedqa-e5-v5 embeddings "
            "with cosine similarity. Second, pass those 10 candidates through the cross-encoder reranker "
            "nvidia/nv-rerankqa-mistral-4b-v3 to select the top-4 chunks before constructing the ChatNVIDIA prompt. "
            "Recommended chunking parameters for technical manuals are chunk_size=500 characters with chunk_overlap=80."
        ),
    },
    {
        "doc_id": "GUARD-SEC-309",
        "title": "NeMo Guardrails & Trustworthy AI Compliance",
        "text": (
            "Customer-facing chatbots must enforce three layers of NeMo Guardrails defined in config.yml and Colang (.co) "
            "files: (1) Input rails for jailbreak detection and PII masking, (2) Dialog rails to keep conversations on "
            "approved technical support topics and refuse competitor pricing speculation, and (3) Output rails for "
            "hallucination self-check and citation verification. All incidents map to the NIST AI RMF Manage function."
        ),
    },
    {
        "doc_id": "TUNE-LORA-412",
        "title": "PEFT and LoRA Fine-Tuning Guidelines",
        "text": (
            "When adapting a foundation model to proprietary formatting or domain terminology with fewer than 10,000 "
            "labeled examples, teams should use Parameter-Efficient Fine-Tuning (PEFT) via Low-Rank Adaptation (LoRA) "
            "with rank r=8 or r=16 and lora_alpha=16 or 32, freezing >99% of base model weights. If GPU VRAM is "
            "constrained to 16GB-24GB, use QLoRA with 4-bit NormalFloat (NF4) base weights and BF16 compute dtype."
        ),
    },
]


def recursive_chunk_documents(docs: List[Dict[str, str]], chunk_size: int = 240, chunk_overlap: int = 50) -> List[DocChunk]:
    chunks: List[DocChunk] = []
    for d in docs:
        text = d["text"]
        start = 0
        idx = 0
        while start < len(text):
            end = min(len(text), start + chunk_size)
            # Snap to sentence or word boundary if not at end of document
            if end < len(text):
                period_pos = text.rfind(". ", start, end)
                space_pos = text.rfind(" ", start, end)
                if period_pos != -1 and period_pos > start + chunk_size // 2:
                    end = period_pos + 1
                elif space_pos != -1 and space_pos > start + chunk_size // 2:
                    end = space_pos
            piece = text[start:end].strip()
            if piece:
                chunks.append(DocChunk(
                    doc_id=d["doc_id"],
                    title=d["title"],
                    chunk_id=f"{d['doc_id']}#c{idx}",
                    text=piece,
                    char_len=len(piece),
                ))
                idx += 1
            if end >= len(text):
                break
            start = max(start + 1, end - chunk_overlap)
    return chunks


corpus_chunks = recursive_chunk_documents(KNOWLEDGE_BASE_DOCS, chunk_size=240, chunk_overlap=50)
print(f"Created {len(corpus_chunks)} overlapping chunks from {len(KNOWLEDGE_BASE_DOCS)} source documents:\n")
for c in corpus_chunks:
    print(f"  [{c.chunk_id:<15}] ({c.char_len:>3} chars): {c.text[:85]}...")

Created 10 overlapping chunks from 4 source documents:

  [NIM-OPS-101#c0 ] (155 chars): NVIDIA NIM containers expose an OpenAI-compatible HTTP REST server on default port 80...
  [NIM-OPS-101#c1 ] (237 chars): endpoints at /v1/health/ready and /v1/health/live. When launching a NIM container wit...
  [NIM-OPS-101#c2 ] (171 chars): m/.cache so TensorRT-LLM engines are cached across container restarts. For Llama-3-8B...
  [RAG-ARCH-204#c0] (216 chars): All production RAG pipelines must use the two-stage retrieval pattern: first retrieve...
  [RAG-ARCH-204#c1] (221 chars): v-embedqa-e5-v5 embeddings with cosine similarity. Second, pass those 10 candidates t...
  [RAG-ARCH-204#c2] (156 chars): chunks before constructing the ChatNVIDIA prompt. Recommended chunking parameters for...
  [GUARD-SEC-309#c0] (233 chars): Customer-facing chatbots must enforce three layers of NeMo Guardrails defined in conf...
  [GUARD-SEC-309#c1] (238 chars): rails to keep conversations on approved technical suppo

---
## 3. Step 2: Reference Book Code — LangChain + NVIDIA Endpoints Skeleton

In **Chapter 3 (*Implementing RAG with Frameworks and NVIDIA Tools*)**, the book presents the canonical LangChain + NVIDIA NIM RAG pipeline using `NVIDIAEmbeddings`, `FAISS`, `NVIDIARerank`, `ContextualCompressionRetriever`, and `ChatNVIDIA`. Let's inspect and define both the live LangChain NVIDIA implementation and our self-contained execution engine so you can run the full pipeline immediately.

In [3]:
# Reference Book Code Pattern (Chapter 3: Implementing RAG with Frameworks and NVIDIA Tools)
BOOK_RAG_SKELETON = '''
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever

# 1. Chunk documents with overlap
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(raw_docs)

# 2. Embed chunks with NVIDIAEmbeddings and index in FAISS
embedder = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5")
vectorstore = FAISS.from_documents(chunks, embedder)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 3. Wrap base retriever with NVIDIARerank cross-encoder
reranker = NVIDIARerank(model="nvidia/nv-rerankqa-mistral-4b-v3", top_n=4)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=base_retriever
)

# 4. Generate grounded answer with ChatNVIDIA
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0.1)
'''
print("Loaded Chapter 3 Canonical LangChain + NVIDIA Endpoints Architecture.")

Loaded Chapter 3 Canonical LangChain + NVIDIA Endpoints Architecture.


---
## 4. Step 3: Building & Running the Two-Stage Vector Index + Cross-Encoder Reranker

Now let's execute the complete RAG pipeline:
1. **Bi-Encoder Dense Embedding (`nvidia/nv-embedqa-e5-v5`):** Converts each chunk and query into an L2-normalized vector $\mathbf{v} \in \mathbb{R}^d$ and computes **Cosine Similarity**:
   $$\text{CosineSimilarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\|_2 \|\mathbf{d}\|_2}$$
2. **Cross-Encoder Reranking (`nvidia/nv-rerankqa-mistral-4b-v3`):** Re-scores the top-$k$ retrieved candidates by evaluating exact query-document term interactions and boosts precision at top-$n$.
3. **Grounded Generation (`meta/llama-3.1-8b-instruct`):** Synthesizes an answer **strictly from the reranked context chunks** with mandatory `[doc_id]` citations, and explicitly abstains when asked an out-of-scope question!

In [4]:
class NVIDIARAGPipeline:
    def __init__(self, chunks: List[DocChunk]):
        self.chunks = chunks
        self.live_api = bool(os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"))
        self._build_vocab_and_embeddings()

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r"[a-z0-9\-\/\.]+", text.lower())

    def _build_vocab_and_embeddings(self):
        # Build IDF-weighted dense concept vectors (or call live NVIDIAEmbeddings if configured)
        doc_tokens = [self._tokenize(c.text + " " + c.title) for c in self.chunks]
        vocab = sorted({tok for toks in doc_tokens for tok in toks})
        self.vocab_idx = {w: i for i, w in enumerate(vocab)}
        n_docs = len(self.chunks)
        df = np.zeros(len(vocab), dtype=np.float32)
        for toks in doc_tokens:
            for w in set(toks):
                df[self.vocab_idx[w]] += 1.0
        self.idf = np.log((n_docs + 1.0) / (df + 1.0)) + 1.0

        self.chunk_matrix = np.zeros((n_docs, len(vocab)), dtype=np.float32)
        for i, toks in enumerate(doc_tokens):
            for w in toks:
                self.chunk_matrix[i, self.vocab_idx[w]] += 1.0
            self.chunk_matrix[i] *= self.idf
            norm = np.linalg.norm(self.chunk_matrix[i]) + 1e-9
            self.chunk_matrix[i] /= norm

    def embed_query(self, query: str) -> np.ndarray:
        vec = np.zeros(len(self.vocab_idx), dtype=np.float32)
        for w in self._tokenize(query):
            if w in self.vocab_idx:
                vec[self.vocab_idx[w]] += 1.0
        vec *= self.idf
        norm = np.linalg.norm(vec) + 1e-9
        return vec / norm

    def retrieve_and_rerank(self, query: str, top_k: int = 5, top_n: int = 3) -> List[Dict]:
        # Stage 1: Vector similarity search (Cosine Similarity on L2-normalized vectors)
        q_vec = self.embed_query(query)
        bi_scores = self.chunk_matrix @ q_vec
        top_k_indices = np.argsort(bi_scores)[::-1][:top_k]

        # Stage 2: Cross-Encoder Reranking (joint query-chunk exact phrase & token interaction scoring)
        q_terms = set(self._tokenize(query))
        candidates = []
        for rank_bi, idx in enumerate(top_k_indices, 1):
            c = self.chunks[idx]
            c_terms = set(self._tokenize(c.text))
            overlap = len(q_terms & c_terms) / max(1, len(q_terms))
            rerank_score = float(0.55 * bi_scores[idx] + 0.45 * overlap)
            candidates.append({
                "chunk_id": c.chunk_id,
                "doc_id": c.doc_id,
                "title": c.title,
                "text": c.text,
                "bi_encoder_rank": rank_bi,
                "bi_encoder_score": round(float(bi_scores[idx]), 4),
                "rerank_score": round(rerank_score, 4),
            })

        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
        for rank_rr, item in enumerate(candidates, 1):
            item["reranker_rank"] = rank_rr
        return candidates[:top_n]

    def answer_question(self, query: str, top_k: int = 5, top_n: int = 3) -> Dict:
        top_chunks = self.retrieve_and_rerank(query, top_k=top_k, top_n=top_n)
        max_score = top_chunks[0]["rerank_score"] if top_chunks else 0.0

        context_block = "\n\n".join(
            f"[{c['doc_id']}] ({c['chunk_id']}): {c['text']}" for c in top_chunks
        )
        system_prompt = (
            "You are a factual NVIDIA Enterprise AI Assistant. Answer the user's question using ONLY "
            "the provided context chunks. Cite every fact using its bracketed document ID like [NIM-OPS-101]. "
            "If the context does not contain the answer, state clearly: "
            "'I cannot answer this question based on the provided knowledge base.'"
        )
        user_prompt = f"Context:\n{context_block}\n\nQuestion: {query}\n\nAnswer:"

        # Prepare deterministic grounded fallback in case live API is not set or returns None
        if max_score < 0.12:
            fallback_text = (
                "I cannot answer this question based on the provided knowledge base. "
                "The retrieved documents cover NVIDIA NIM deployment, RAG architecture, NeMo Guardrails, and LoRA fine-tuning, "
                "but do not contain information about this topic."
            )
        else:
            best = top_chunks[0]
            second = top_chunks[1] if len(top_chunks) > 1 and top_chunks[1]["rerank_score"] > 0.15 else None
            fallback_text = f"Based on the enterprise documentation, {best['text']} [{best['doc_id']}]."
            if second and second["doc_id"] != best["doc_id"]:
                fallback_text += f" Additionally, {second['text']} [{second['doc_id']}]."

        answer_text = None
        if self.live_api:
            try:
                from openai import OpenAI
                client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"], timeout=60.0)
                stream = client.chat.completions.create(
                    model="google/gemma-4-31b-it",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=0.1,
                    max_tokens=1024,
                    stream=True,
                )
                c_pieces, r_pieces = [], []
                for chunk in stream:
                    if not getattr(chunk, "choices", None):
                        continue
                    delta = getattr(chunk.choices[0], "delta", None)
                    if delta is None:
                        continue
                    if getattr(delta, "content", None):
                        c_pieces.append(str(delta.content))
                    elif getattr(delta, "reasoning_content", None):
                        r_pieces.append(str(delta.reasoning_content))
                time.sleep(1.5)
                answer_text = "".join(c_pieces).strip() or "".join(r_pieces).strip()
            except Exception as e:
                print(f"[WARN] Live NIM RAG call fallback ({type(e).__name__})")

        if not answer_text:
            answer_text = fallback_text

        citations = re.findall(r"\[([A-Z]+-[A-Z]+-\d+)\]", answer_text)
        return {
            "query": query,
            "retrieved_chunks": top_chunks,
            "answer": answer_text,
            "citations": sorted(set(citations)),
            "abstained": "cannot answer" in answer_text.lower(),
        }


rag_bot = NVIDIARAGPipeline(corpus_chunks)
print("RAG Pipeline initialized and ready.")

RAG Pipeline initialized and ready.


---
## 5. Step 4: Testing In-Scope vs. Out-of-Scope Queries

In **Chapter 3 & Chapter 6**, a trustworthy RAG system must pass two complementary tests:
1. **In-Scope Queries:** Retrieve the right chunks, answer accurately, and include verifiable `[doc-id]` citations.
2. **Out-of-Scope Queries:** When asked about something absent from the knowledge base (e.g., *"What is the cafeteria lunch menu on Fridays?"* or *"What will NVIDIA's stock price be in 2029?"*), the system must **abstain** rather than hallucinate!

In [5]:
test_queries = [
    ("IN-SCOPE 1", "What port does an NVIDIA NIM container expose by default, and what Docker flags are required?"),
    ("IN-SCOPE 2", "Which embedding model and cross-encoder reranker model are required by our enterprise RAG standard?"),
    ("IN-SCOPE 3", "When should teams use LoRA or QLoRA, and what rank and precision settings are recommended?"),
    ("OUT-OF-SCOPE", "What is the company's reimbursement limit for international hotel stays in Tokyo?"),
]

for q_type, query in test_queries:
    res = rag_bot.answer_question(query, top_k=5, top_n=2)
    print("=" * 85)
    print(f"[{q_type}] Query: {query}")
    print("-" * 85)
    for c in res["retrieved_chunks"]:
        print(f"  Retrieved {c['chunk_id']:<15} | Bi-Encoder Score: {c['bi_encoder_score']:.3f} | Rerank Score: {c['rerank_score']:.3f}")
    print(f"\nGrounded Answer:\n  {res['answer']}")
    print(f"Extracted Citations: {res['citations']} | Abstained (Out-of-Scope Protection): {res['abstained']}\n")

[IN-SCOPE 1] Query: What port does an NVIDIA NIM container expose by default, and what Docker flags are required?
-------------------------------------------------------------------------------------
  Retrieved NIM-OPS-101#c0  | Bi-Encoder Score: 0.525 | Rerank Score: 0.499
  Retrieved NIM-OPS-101#c1  | Bi-Encoder Score: 0.332 | Rerank Score: 0.333

Grounded Answer:
  Based on the enterprise documentation, NVIDIA NIM containers expose an OpenAI-compatible HTTP REST server on default port 8000 and health check endpoints at /v1/health/ready and /v1/health/live. [NIM-OPS-101].
Extracted Citations: ['NIM-OPS-101'] | Abstained (Out-of-Scope Protection): False

[IN-SCOPE 2] Query: Which embedding model and cross-encoder reranker model are required by our enterprise RAG standard?
-------------------------------------------------------------------------------------
  Retrieved RAG-ARCH-204#c1 | Bi-Encoder Score: 0.241 | Rerank Score: 0.202
  Retrieved TUNE-LORA-412#c1 | Bi-Encoder Score: 0.12

---
## 6. Hands-On Exercises

### Exercise 1: Chunk Size & Overlap Sensitivity Experiment
Re-chunk `KNOWLEDGE_BASE_DOCS` with:
- Configuration A: `chunk_size=120, chunk_overlap=0` (tiny chunks with zero overlap)
- Configuration B: `chunk_size=240, chunk_overlap=50` (balanced chunks with ~20% overlap)
Compare the number of chunks produced and check whether any sentence in Configuration A gets split across two chunks without overlapping context.

In [6]:
# TODO: Complete Exercise 1

### Exercise 2: Citation Faithfulness Verifier
Write a function `verify_rag_citations(result: dict) -> bool` that checks:
1. If `result["abstained"]` is `True`, no citations are required (`True`).
2. Otherwise, every `[DOC-ID]` cited in `result["answer"]` must actually appear in the `doc_id` list of `result["retrieved_chunks"]` (preventing **hallucinated citations**!).

In [ ]:
# TODO: Complete Exercise 2
def verify_rag_citations(result: dict) -> bool:
    pass

---
## 7. Reference Solutions

In [7]:
# Solution to Exercise 1: Chunk Size & Overlap Comparison
chunks_no_overlap = recursive_chunk_documents(KNOWLEDGE_BASE_DOCS, chunk_size=120, chunk_overlap=0)
chunks_with_overlap = recursive_chunk_documents(KNOWLEDGE_BASE_DOCS, chunk_size=240, chunk_overlap=50)
print(f"Config A (size=120, overlap=0)  -> {len(chunks_no_overlap)} chunks (risk of splitting multi-clause facts!)")
print(f"Config B (size=240, overlap=50) -> {len(chunks_with_overlap)} chunks (preserves boundary context)")

# Solution to Exercise 2: Citation Faithfulness Verifier
def verify_rag_citations_solution(result: dict) -> bool:
    if result["abstained"]:
        return True
    retrieved_ids = {c["doc_id"] for c in result["retrieved_chunks"]}
    cited_ids = set(result["citations"])
    return bool(cited_ids) and cited_ids.issubset(retrieved_ids)

sample_res = rag_bot.answer_question("What layers of NeMo Guardrails are required for chatbots?")
print("\n=== Exercise 2: Citation Verification ===")
print("Answer        :", sample_res["answer"])
print("Cited IDs     :", sample_res["citations"])
print("Retrieved IDs :", [c["doc_id"] for c in sample_res["retrieved_chunks"]])
print("Faithful?     :", verify_rag_citations_solution(sample_res))

Config A (size=120, overlap=0)  -> 17 chunks (risk of splitting multi-clause facts!)
Config B (size=240, overlap=50) -> 10 chunks (preserves boundary context)

=== Exercise 2: Citation Verification ===
Answer        : Based on the enterprise documentation, Customer-facing chatbots must enforce three layers of NeMo Guardrails defined in config.yml and Colang (.co) files: (1) Input rails for jailbreak detection and PII masking, (2) Dialog rails to keep conversations on approved technical [GUARD-SEC-309]. Additionally, m/.cache so TensorRT-LLM engines are cached across container restarts. For Llama-3-8B-Instruct in FP16 precision, a minimum of 1x NVIDIA L4 (24GB) or A10G GPU is required. [NIM-OPS-101].
Cited IDs     : ['GUARD-SEC-309', 'NIM-OPS-101']
Retrieved IDs : ['GUARD-SEC-309', 'NIM-OPS-101', 'RAG-ARCH-204']
Faithful?     : True


---
## 8. NCA-GENL Chapter 3 Review Takeaways

- **RAG vs. Fine-Tuning:** Choose **RAG** for dynamic facts, proprietary knowledge bases, and source citations (`[doc-id]`); choose **Fine-Tuning (LoRA)** to adapt style, tone, or task formatting.
- **Chunking Parameters:** `chunk_size` controls context granularity; `chunk_overlap` prevents losing context at chunk boundaries.
- **Two-Stage Retrieval:** Fast bi-encoder vector search (`NVIDIAEmbeddings` + `FAISS`) retrieves top-$k$ candidates; cross-encoder reranking (`NVIDIARerank`) re-scores those candidates jointly with the query to maximize top-$n$ precision.